# Deep FBSDE Solver: High-Dimensional Option Pricing via PyTorch

**Forward-Backward Stochastic Neural Networks (FBSNNs)** based on Raissi (2018) and Han, Jentzen, E (2018).
Solves high-dimensional non-linear parabolic PDEs and Black-Scholes equations without spatial meshes.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import norm
from fbsde_solver import BlackScholesCallFBSNN, black_scholes_analytical_call

## 1. Parameters & Analytical Benchmark

In [ ]:
S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.25
bs_price = black_scholes_analytical_call(S0, K, T, r, sigma)
print(f"Analytical Black-Scholes Price: ${bs_price:.4f}")

## 2. Train Forward-Backward Stochastic Neural Network

In [ ]:
solver = BlackScholesCallFBSNN(S0=1.0, K=1.0, T=T, r=r, sigma=sigma, N=30, M=64, layers=[2, 64, 64, 1])
losses = []
prices = []
for it in range(1, 401):
    loss, y0 = solver.train_step()
    losses.append(loss)
    prices.append(y0 * S0)
print(f"Final Estimated Price: ${prices[-1]:.4f} (BS Benchmark: ${bs_price:.4f})")

## 3. Convergence Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(losses, color="firebrick", lw=1.5)
ax1.set_title("FBSNN Terminal Loss Convergence")
ax1.set_xlabel("Iteration"); ax1.set_ylabel("MSE Loss"); ax1.grid(True, alpha=0.3)

ax2.plot(prices, label="FBSNN Predicted Price", color="navy", lw=1.5)
ax2.axhline(bs_price, color="darkgreen", linestyle="--", label=f"Black-Scholes Benchmark (${bs_price:.2f})")
ax2.set_title("Option Price Convergence")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Price ($)"); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()